# MPro Activity Predictions — M1 Model Demo

> **Run this to see M1's activity predictions on the full 2,025-compound dataset.
> This is what feeds M2's ranking and M6's docking.**

This notebook runs inference with the trained Random Forest model
(`models/mpro_activity_rf_v1.pkl`) on the MPro-targeted compound library.
It is read-only — no model training, no data mutation.

**Sections:**
1. Imports & model load
2. Sample predictions (10 compounds)
3. Batch prediction over all 2,025 compounds
4. Top-15 predicted actives (M6 docking shortlist)
5. Model context & limitations


## 1. Imports & Model Load

In [ ]:
import os, sys, json, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

warnings.filterwarnings("ignore")

# Make src importable regardless of cwd
ROOT = os.path.abspath("..")  # notebook lives in notebooks/, project root is one level up
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src.ml.predict import predict_activity, MODEL_VERSION

# ── Load model & metadata ──────────────────────────────────────────────────
MODEL_PATH = os.path.join(ROOT, "models", "mpro_activity_rf_v1.pkl")
META_PATH  = os.path.join(ROOT, "data", "processed", "feature_metadata.json")
CSV_PATH   = os.path.join(ROOT, "data", "raw", "mpro_labeled_smiles.csv")

with open(MODEL_PATH, "rb") as f:
    model = pickle.load(f)

with open(META_PATH) as f:
    meta = json.load(f)

print(f"Model loaded   : {MODEL_PATH}")
print(f"Model version  : {MODEL_VERSION}")
print(f"n_estimators   : {meta.get('n_estimators', model.n_estimators)}")
print(f"n_features     : {meta['n_features']}")
print(f"Training split : {meta['train_frac']*100:.0f}% train / {(1-meta['train_frac'])*100:.0f}% test (scaffold)")


## 2. Sample Predictions — 10 Random Compounds

5 known **Actives** + 5 known **Inactives** drawn from the labeled dataset.
The model was trained on a scaffold split — some of these compounds may have
been in the training set (all 2,025 are scored here for illustration).


In [ ]:
df_all = pd.read_csv(CSV_PATH)

# Pick 5 actives + 5 inactives reproducibly
rng = np.random.default_rng(42)
actives   = df_all[df_all["activity_label"] == "active"].sample(5,  random_state=42)
inactives = df_all[df_all["activity_label"] == "inactive"].sample(5, random_state=42)
sample    = pd.concat([actives, inactives]).reset_index(drop=True)

# Predict
preds = predict_activity(sample["smiles"].tolist())

# Merge
sample_out = sample[["compound_id", "smiles", "activity_label"]].copy()
sample_out = sample_out.merge(preds[["smiles", "predicted_activity_score", "predicted_class"]],
                              on="smiles", how="left")
sample_out.columns = ["compound_id", "smiles", "true_label",
                      "predicted_score", "predicted_class"]
sample_out["correct"] = sample_out["true_label"].str.lower() == sample_out["predicted_class"].str.lower()

# Display
pd.set_option("display.max_colwidth", 50)
display(sample_out[["compound_id", "true_label", "predicted_score",
                     "predicted_class", "correct"]]
        .style
        .format({"predicted_score": "{:.4f}"})
        .applymap(lambda v: "background-color: #d4edda" if v else "background-color: #f8d7da",
                  subset=["correct"]))


## 3. Batch Prediction — All 2,025 Compounds

Runs `predict_activity()` on every compound. Should complete in < 5 seconds.


In [ ]:
import time

t0 = time.time()
all_preds = predict_activity(df_all["smiles"].tolist())
elapsed   = time.time() - t0

# Merge true labels
all_preds["true_label"] = df_all["activity_label"].values
all_preds["compound_id"] = df_all["compound_id"].values

n_pred_active   = (all_preds["predicted_class"] == "Active").sum()
n_pred_inactive = (all_preds["predicted_class"] == "Inactive").sum()

print(f"Inference time      : {elapsed:.2f}s  ({len(all_preds)/elapsed:.0f} compounds/s)")
print(f"Predicted Active    : {n_pred_active:>5}  ({n_pred_active/len(all_preds)*100:.1f}%)")
print(f"Predicted Inactive  : {n_pred_inactive:>5}  ({n_pred_inactive/len(all_preds)*100:.1f}%)")
print(f"Parse failures      : {len(df_all) - len(all_preds)}")


In [ ]:
# ── Score distribution histogram ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

colors = {"active": "#e74c3c", "inactive": "#3498db"}

for ax, (label, grp) in zip(axes, all_preds.groupby("true_label")):
    ax.hist(grp["predicted_activity_score"], bins=30,
            color=colors.get(label, "#7f8c8d"), alpha=0.85, edgecolor="white")
    ax.set_title(f"Score distribution — True {label.capitalize()}s (n={len(grp)})",
                 fontsize=12, fontweight="bold")
    ax.set_xlabel("Predicted Activity Score (P(Active))", fontsize=10)
    ax.set_ylabel("Compound Count", fontsize=10)
    ax.axvline(0.5, color="black", linestyle="--", linewidth=1.2, label="Threshold=0.5")
    ax.legend()
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(ROOT, "results", "score_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved → results/score_distribution.png")


In [ ]:
# ── Confusion matrix (all 2025 compounds) ─────────────────────────────────
y_true = (all_preds["true_label"] == "active").astype(int)
y_pred = (all_preds["predicted_class"] == "Active").astype(int)

cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(cm, display_labels=["Inactive", "Active"])
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Confusion Matrix — All 2,025 Compounds\n"
             "(includes training set; for test-only metrics see model_metrics_v1.json)",
             fontsize=10, pad=12)
plt.tight_layout()
plt.savefig(os.path.join(ROOT, "results", "confusion_matrix_full.png"), dpi=150, bbox_inches="tight")
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}")
print(f"Precision (all): {tp/(tp+fp):.3f}   Recall (all): {tp/(tp+fn):.3f}")
print()
print("NOTE: This matrix covers all 2,025 compounds — train + test.")
print("For held-out test-set metrics, see results/model_metrics_v1.json")


## 4. Top-15 Predicted Actives — M6 Docking Shortlist

These are the compounds the model is most confident are Active against MPro.
In the full pipeline, this list feeds into M6 (molecular docking) for
structural confirmation.


In [ ]:
top15 = (all_preds
         .sort_values("predicted_activity_score", ascending=False)
         .head(15)
         [["compound_id", "smiles", "predicted_activity_score", "predicted_class", "true_label"]]
         .reset_index(drop=True))

top15.index = top15.index + 1  # 1-based rank
top15.index.name = "rank"

display(top15.style
        .format({"predicted_activity_score": "{:.4f}"})
        .bar(subset=["predicted_activity_score"], color="#e74c3c", vmin=0, vmax=1)
        .applymap(lambda v: "color: #e74c3c; font-weight: bold" if v == "active" else "color: #3498db",
                  subset=["true_label"]))

n_top_correct = (top15["true_label"] == "active").sum()
print(f"\nOf the top 15 predicted actives, {n_top_correct} are true Actives in the dataset.")


## 5. Model Context & Limitations

### What this model is
- **Algorithm**: Random Forest (100 trees, balanced class weights, scaffold split)
- **Features**: Morgan fingerprint (radius=2, 2048 bits) + 6 physicochemical descriptors
  (MW, LogP, TPSA, HBD, HBA, Rotatable Bonds) = **2,054-dim feature vector**
- **Training data**: 405 Actives + 1,620 Inactives from PubChem AID 1706 (MPro assay)
- **Split**: Murcko scaffold-based 80/20 — scaffold groups held out to reduce leakage

### Baseline performance (held-out test set only)
| Metric | Value |
|--------|-------|
| ROC-AUC | 0.766 |
| PR-AUC | 0.466 |
| Precision | 0.813 |
| Recall | 0.210 |
| F1 | 0.333 |

### Known limitations
- **Low recall**: At threshold=0.5 the model misses ~79% of true Actives.
  This is expected with qualitative assay data and class imbalance.
- **Qualitative labels only**: AID 1706 uses binary Active/Inactive labels —
  no IC50/EC50 quantitative potency data is available for most compounds.
- **Small training set**: 405 actives is limited; more actives would improve recall.
- **Baseline only**: No hyperparameter tuning has been performed yet.

> See `LIMITATIONS.md` (project root) for full documentation of model constraints
> and intended use within the pipeline.

### Role in the pipeline
```
M1 (this model) → M2 (ranking) → M6 (docking)
```
M1 scores all candidate compounds → M2 re-ranks by combining M1 score with
other signals → M6 docks the top shortlist against the MPro crystal structure.
